In [0]:
# ============================================================
# 00_PIPELINE_CONTROL
# Objectif :
# - Lire la version du GTFS depuis feed_info.txt
# - Créer la table de contrôle si elle n'existe pas
# - Vérifier si ce feed a déjà été traité avec succès
# - Retourner NEW_FEED ou ALREADY_PROCESSED
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

FEED_INFO_PATH = "/Volumes/workspace/sncf_bronze/landing/feed_info.txt"
CONTROL_TABLE = "workspace.sncf_bronze.pipeline_control"


# ============================================================
# 2. CRÉATION DE LA TABLE DE CONTRÔLE SI NÉCESSAIRE
# ============================================================

if not spark.catalog.tableExists(CONTROL_TABLE):

    spark.sql(f"""
        CREATE TABLE {CONTROL_TABLE} (
            feed_start_date DATE,
            feed_end_date DATE,
            processed_at TIMESTAMP,
            status STRING
        )
        USING DELTA
    """)

    print("Table pipeline_control créée.")

else:
    print("Table pipeline_control existe déjà.")


# ============================================================
# 3. LECTURE DU FEED_INFO
# ============================================================

df_feed = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(FEED_INFO_PATH)
)


# ============================================================
# 4. RÉCUPÉRATION DE LA VERSION DU FEED
# ============================================================

current_feed = (
    df_feed
    .select(
        F.to_date(F.col("feed_start_date"), "yyyyMMdd")
            .alias("feed_start_date"),

        F.to_date(F.col("feed_end_date"), "yyyyMMdd")
            .alias("feed_end_date")
    )
    .first()
)

if current_feed is None:
    raise Exception("feed_info.txt est vide.")


feed_start = current_feed["feed_start_date"]
feed_end = current_feed["feed_end_date"]


if feed_start is None or feed_end is None:
    raise Exception(
        "Impossible de récupérer feed_start_date ou feed_end_date "
        "depuis feed_info.txt."
    )


print("==========================================")
print("GTFS détecté")
print("Feed start :", feed_start)
print("Feed end   :", feed_end)
print("==========================================")


# ============================================================
# 5. VÉRIFICATION DANS PIPELINE_CONTROL
# ============================================================

already_processed = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("feed_start_date") == F.lit(feed_start)) &
        (F.col("feed_end_date") == F.lit(feed_end)) &
        (F.col("status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)


# ============================================================
# 6. DÉCISION
# ============================================================

if already_processed:

    pipeline_status = "ALREADY_PROCESSED"

    print("==========================================")
    print("GTFS DÉJÀ TRAITÉ")
    print("Aucune nouvelle ingestion nécessaire.")
    print("==========================================")

else:

    pipeline_status = "NEW_FEED"

    print("==========================================")
    print("NOUVEAU GTFS")
    print("Le pipeline Bronze -> Silver -> Gold doit être exécuté.")
    print("==========================================")


# ============================================================
# 7. VALEUR À UTILISER PAR LE WORKFLOW
# ============================================================

dbutils.jobs.taskValues.set(
    key="pipeline_status",
    value=pipeline_status
)

dbutils.jobs.taskValues.set(
    key="feed_start_date",
    value=str(feed_start)
)

dbutils.jobs.taskValues.set(
    key="feed_end_date",
    value=str(feed_end)
)


print("pipeline_status =", pipeline_status)

Table pipeline_control créée.
GTFS détecté
Feed start : 2026-09-03
Feed end   : 2027-02-28
NOUVEAU GTFS
Le pipeline Bronze -> Silver -> Gold doit être exécuté.
pipeline_status = NEW_FEED
